# Research reproduction: Othello-GPT world representations

This notebook uses the released Othello-GPT checkpoint, linear probe, and game sequences to reproduce named quantitative results from [*Linear Latent World Models in Simple Transformers*](https://arxiv.org/abs/2310.07582), then checks that a TDHook intervention is numerically equivalent to the authors' native TransformerLens hook.

The scientific pipeline is **behavior -> representation decoding -> intervention parity -> aggregate causal validation**. The last stage is deliberately marked incomplete here: matching a hook is an API result, not yet a reproduction of the paper's 50-game inverse-map intervention sweep. That remaining work is tracked in [issue #107](https://github.com/Xmaster6y/tdhook/issues/107).


## Frozen sources and acceptance gates

Paper targets: 99.5% Mine/Yours decoding accuracy at deep layers (Table 1) and 99.9% legal next moves for the 8-layer model (Appendix B). We preregister an absolute tolerance of 0.5 percentage points for this 100-game released-data slice. The [released implementation and assets](https://github.com/likenneth/othello_world) are pinned to immutable revisions and checked before execution.


In [1]:
from __future__ import annotations

import hashlib
import importlib.util
import json
import sys
import types
import urllib.request
from pathlib import Path

import numpy as np
import torch
from huggingface_hub import hf_hub_download
from transformer_lens import HookedTransformer, HookedTransformerConfig

from tdhook.session import HookSession
from tdhook.targets import Target

OTHELLO_REVISION = "f23bb5696cf30b93bd8af8a391ee33fc3aac417e"
HF_REVISION = "905ca1a68b9f7dff77adc56af1962e5f6fcac274"
EXPECTED = {
    "main_linear_probe.pth": "eb8ab01a70e63305661c97a211a1321decc16451baff909481cde3b615a7fb16",
    "board_seqs_int_small.npy": "fb11a862c50f933e029e0708d5266e838fcc3dad8021cbc6d9ec964dfedb2f77",
    "board_seqs_string_small.npy": "99565a9962ed79064f0e114224325c81599a8f03e5e6b8ea6cecdb0df687aa37",
    "othello.py": "b2d7fa23a0b6a84daef99d3d26e4f926386fa03986db4fe7307d1baba34a47e7",
    "synthetic_model.pth": "aef97afb1e7ec9dbef1d9007174458830a5c428d62f1e2338128879b179bd5c4",
}
CACHE = Path.home() / ".cache" / "tdhook" / "othello-reproduction"
CACHE.mkdir(parents=True, exist_ok=True)
BASE = f"https://raw.githubusercontent.com/likenneth/othello_world/{OTHELLO_REVISION}"
ASSETS = {
    "main_linear_probe.pth": f"{BASE}/mechanistic_interpretability/main_linear_probe.pth",
    "board_seqs_int_small.npy": f"{BASE}/mechanistic_interpretability/board_seqs_int_small.npy",
    "board_seqs_string_small.npy": f"{BASE}/mechanistic_interpretability/board_seqs_string_small.npy",
    "othello.py": f"{BASE}/data/othello.py",
}


def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def fetch(name: str) -> Path:
    path = CACHE / name
    if not path.exists():
        urllib.request.urlretrieve(ASSETS[name], path)
    observed = sha256(path)
    expected = EXPECTED[name]
    if expected != "TO_BE_FILLED" and observed != expected:
        raise RuntimeError(f"{name} checksum mismatch: {observed}")
    return path


paths = {name: fetch(name) for name in ASSETS}
model_path = Path(
    hf_hub_download(
        repo_id="NeelNanda/Othello-GPT-Transformer-Lens",
        filename="synthetic_model.pth",
        revision=HF_REVISION,
        token=False,
    )
)
assert sha256(model_path) == EXPECTED["synthetic_model.pth"]
manifest = {
    "othello_revision": OTHELLO_REVISION,
    "hf_revision": HF_REVISION,
    "assets": {name: sha256(path) for name, path in paths.items()},
    "model_sha256": sha256(model_path),
    "torch": torch.__version__,
}
print(json.dumps(manifest, indent=2))

{
  "othello_revision": "f23bb5696cf30b93bd8af8a391ee33fc3aac417e",
  "hf_revision": "905ca1a68b9f7dff77adc56af1962e5f6fcac274",
  "assets": {
    "main_linear_probe.pth": "eb8ab01a70e63305661c97a211a1321decc16451baff909481cde3b615a7fb16",
    "board_seqs_int_small.npy": "fb11a862c50f933e029e0708d5266e838fcc3dad8021cbc6d9ec964dfedb2f77",
    "board_seqs_string_small.npy": "99565a9962ed79064f0e114224325c81599a8f03e5e6b8ea6cecdb0df687aa37",
    "othello.py": "b2d7fa23a0b6a84daef99d3d26e4f926386fa03986db4fe7307d1baba34a47e7"
  },
  "model_sha256": "aef97afb1e7ec9dbef1d9007174458830a5c428d62f1e2338128879b179bd5c4",
  "torch": "2.7.1"
}


In [2]:
# The released simulator imports `pgn` only for championship-data utilities.
# This reproduction uses synthetic games, so a stub keeps that unrelated optional dependency out.
sys.modules.setdefault("pgn", types.ModuleType("pgn"))
spec = importlib.util.spec_from_file_location("released_othello", paths["othello.py"])
othello = importlib.util.module_from_spec(spec)
assert spec.loader is not None
spec.loader.exec_module(othello)

device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
cfg = HookedTransformerConfig(
    n_layers=8,
    d_model=512,
    d_head=64,
    n_heads=8,
    d_mlp=2048,
    d_vocab=61,
    n_ctx=59,
    act_fn="gelu",
    normalization_type="LNPre",
    device=str(device),
)
model = HookedTransformer(cfg)
model.load_state_dict(torch.load(model_path, map_location=device, weights_only=False))
model.eval()
probe = torch.load(paths["main_linear_probe.pth"], map_location=device, weights_only=False)
games_int = torch.from_numpy(np.load(paths["board_seqs_int_small.npy"])[:100, :59].astype(np.int64)).to(device)
games_string = np.load(paths["board_seqs_string_small.npy"])[:100, :59]
print({"device": str(device), "games": tuple(games_int.shape), "probe": tuple(probe.shape)})

{'device': 'mps', 'games': (100, 59), 'probe': (3, 512, 8, 8, 3)}


## Stage 1-2: behavior and representation decoding

TDHook captures the complete layer-7 residual stream. The released linear probe then decodes every square as empty, mine, or yours. The parity-specific probes alternate with the side to move; evaluating each on its designated positions follows the released analysis.


In [3]:
states = []
for sequence in games_string:
    board = othello.OthelloBoardState()
    sequence_states = []
    for move in sequence.tolist():
        board.umpire(int(move))
        sequence_states.append(board.state.copy())
    states.append(np.stack(sequence_states))
states = torch.from_numpy(np.stack(states))
labels = torch.empty(states.shape, dtype=torch.long)
labels[states == 0], labels[states == -1], labels[states == 1] = 0, 1, 2
labels = labels.to(device)

residual_target = Target("blocks.6.hook_resid_post", "activation", -1, tuple(range(512)))
with torch.inference_mode():
    with HookSession(model) as session:
        captured = session.capture(residual_target)
        logits = model(games_int)
    probe_logits = torch.einsum("bpd,mdrco->mbprco", captured.value, probe)
    probe_correct = probe_logits.argmax(-1) == labels.unsqueeze(0)

lo, hi = 5, 54
odd_accuracy = float(probe_correct[0, :, lo:hi:2].float().mean().cpu())
even_accuracy = float(probe_correct[1, :, lo + 1 : hi : 2].float().mean().cpu())
designated_accuracy = (odd_accuracy + even_accuracy) / 2

playable = [index for index in range(64) if index not in (27, 28, 35, 36)]
illegal = total = 0
predicted_tokens = logits.argmax(-1).cpu().numpy()
for game_index, sequence in enumerate(games_string):
    board = othello.OthelloBoardState()
    for position, move in enumerate(sequence.tolist()):
        board.umpire(int(move))
        if position < 58:
            token = int(predicted_tokens[game_index, position])
            predicted_square = -1 if token == 0 else playable[token - 1]
            illegal += int(predicted_square not in board.get_valid_moves())
            total += 1
legal_rate = 1 - illegal / total

paper_targets = {"designated_probe_accuracy": 0.995, "legal_move_rate": 0.999}
observed = {
    "odd_probe_accuracy": odd_accuracy,
    "even_probe_accuracy": even_accuracy,
    "designated_probe_accuracy": designated_accuracy,
    "legal_move_rate": legal_rate,
    "evaluated_next_move_positions": total,
}
tolerance = 0.005
gates = {key: abs(observed[key] - target) <= tolerance for key, target in paper_targets.items()}
print(
    json.dumps(
        {"observed": observed, "paper_targets": paper_targets, "tolerance": tolerance, "gates": gates}, indent=2
    )
)
assert all(gates.values())

{
  "observed": {
    "odd_probe_accuracy": 0.9927937388420105,
    "even_probe_accuracy": 0.9928125143051147,
    "designated_probe_accuracy": 0.9928031265735626,
    "legal_move_rate": 0.9993103448275862,
    "evaluated_next_move_positions": 5800
  },
  "paper_targets": {
    "designated_probe_accuracy": 0.995,
    "legal_move_rate": 0.999
  },
  "tolerance": 0.005,
  "gates": {
    "designated_probe_accuracy": true,
    "legal_move_rate": true
  }
}


## Stage 3: intervention implementation parity

For one released game, we capture a single residual-stream position, translate it along a released probe direction, and replace it through `HookSession`. We compare the entire log-probability tensor with the authors' native TransformerLens hook. This gate validates operation equivalence only; it does not establish the paper's aggregate causal claim.


In [4]:
game = games_int[8:9, :51]
position, layer, row, column, scale = 50, 3, 7, 4, 4
direction = probe[0, :, row, column, 2] - probe[0, :, row, column, 1]
direction = direction / direction.norm()
position_target = Target(f"blocks.{layer}.hook_resid_post", "activation", -2, (position,))

with torch.inference_mode():
    with HookSession(model) as capture_session:
        activation = capture_session.capture(position_target)
        model(game)
    coefficient = (activation.value @ direction).unsqueeze(-1)
    replacement = activation.value - (scale + 1) * coefficient * direction
    with HookSession(model) as intervention_session:
        intervention_session.replace(position_target, replacement)
        tdhook_log_probs = model(game).log_softmax(-1)

    def reference_hook(residual, hook):
        coefficient = residual[0, position] @ direction
        residual[0, position] -= (scale + 1) * coefficient * direction

    reference_log_probs = model.run_with_hooks(
        game, fwd_hooks=[(f"blocks.{layer}.hook_resid_post", reference_hook)]
    ).log_softmax(-1)

max_abs_difference = float((tdhook_log_probs - reference_log_probs).abs().max().cpu())
parity_tolerance = 1e-5
print(
    {
        "max_abs_difference": max_abs_difference,
        "tolerance": parity_tolerance,
        "passed": max_abs_difference <= parity_tolerance,
    }
)
print(intervention_session.program)
assert max_abs_difference <= parity_tolerance

{'max_abs_difference': 5.7220458984375e-06, 'tolerance': 1e-05, 'passed': True}
HookProgram(hooks=(HookSpec(module_path='blocks.3.hook_resid_post', operation='replace', direction='fwd', prepend=False, target=Target(module_path='blocks.3.hook_resid_post', kind='activation', feature_axis=-2, indices=(50,), parameter=None, output_path=())),), stopped_at=None)


## Evidence boundary

This notebook reaches **numerical reproduction** for the behavioral and probe results and **API parity** for one causal intervention. It does not yet reach scientific reproduction of Figure 4: that requires the released inverse-map optimization, the 50-game layer/game-length sweep, sham and randomized-probe controls, and aggregate uncertainty. The missing differentiable activation-optimization contract is tracked in [issue #110](https://github.com/Xmaster6y/tdhook/issues/110).
